In [1]:
%pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.5/961.5 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 111.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [2]:
from google.colab import drive, runtime
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import math
import os
import random
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from PIL import Image
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
import xml.etree.ElementTree as ET
import numpy as np
from torchvision.ops import box_iou
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torchvision
from torchvision.transforms import functional as F
from torch.utils.data import Dataset
from torchmetrics.detection import MeanAveragePrecision
from torchvision.models import VGG16_Weights
import warnings
from sklearn.model_selection import train_test_split

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.enabled = True
    torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def get_paths(names, cfg):
    imgs, xmls = [], []
    for name in names:
        img_path = os.path.join(cfg["project_root"], "data", "images", f"{name}.jpg")
        xml_path = os.path.join(cfg["project_root"], "data", "annotations", "xmls", f"{name}.xml")

        img_exists = os.path.isfile(img_path)
        xml_exists = os.path.isfile(xml_path)

        if not img_exists and not xml_exists:
            warnings.warn(f"[SKIP] neither image nor xml found for: {name}")
            continue
        if not img_exists:
            warnings.warn(f"[SKIP] image not found: {img_path}")
            continue
        if not xml_exists:
            warnings.warn(f"[SKIP] annotation not found: {xml_path}")
            continue

        # 둘 다 존재할 때만 append
        imgs.append(img_path)
        xmls.append(xml_path)

    return imgs, xmls

def get_class_map(xml_dir):
    breeds = set()
    for fname in os.listdir(xml_dir):
        if not fname.endswith(".xml"):
            continue
        # 예: 'shiba_inu_156.xml' → 'shiba_inu'
        breed = "_".join(fname.split("_")[:-1]).lower()
        breeds.add(breed)

    breeds = sorted(breeds)
    return {breed: idx for idx, breed in enumerate(breeds)}

def parse_voc_xml(xml_path, class_map):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    #    - os.path.basename: "american_bulldog_130.xml"
    #    - rsplit('.', 1)[0]: "american_bulldog_130"
    #    - rsplit('_', 1)[0]: "american_bulldog"
    basename = os.path.basename(xml_path)
    name_wo_ext = basename.rsplit('.', 1)[0]
    breed = name_wo_ext.rsplit('_', 1)[0].lower()
    label = class_map[breed] + 1  # 배경=0, 실제 클래스는 1부터

    boxes, labels = [], []
    for obj in root.findall("object"):
        bnd = obj.find("bndbox")
        xmin = int(bnd.find("xmin").text)
        ymin = int(bnd.find("ymin").text)
        xmax = int(bnd.find("xmax").text)
        ymax = int(bnd.find("ymax").text)
        boxes.append([xmin, ymin, xmax, ymax])
        labels.append(label)

    return boxes, labels



class PetFaceDataset(Dataset):
    def __init__(self, image_paths, xml_paths, class_map):
        self.image_paths = image_paths
        self.xml_paths   = xml_paths
        self.class_map   = class_map
        self.to_tensor = torchvision.transforms.ToTensor()


    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        boxes, labels = parse_voc_xml(self.xml_paths[idx], self.class_map)

        boxes  = np.array(boxes,  dtype=np.float32)
        labels = np.array(labels, dtype=np.int64)

        img = self.to_tensor(img)

        target = {
            "boxes":    torch.tensor(boxes,  dtype=torch.float32),
            "labels":   torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx])
        }
        return img, target

    def __len__(self):
        return len(self.image_paths)

def get_ssd_model(num_classes: int = 38, pretrained: bool = True, device: torch.device = torch.device("cuda")):
    model = ssd300_vgg16(
        weights=None,                                    # 전체 detection 가중치 사용 안 함
        weights_backbone=VGG16_Weights.IMAGENET1K_V1,     # VGG16 백본만 ImageNet 가중치로 로드
        num_classes=num_classes                           # 원하는 클래스 수 지정
    )
    return model.to(device)

def collate_fn(batch):
    images = []
    targets = []
    for img, target in batch:
        images.append(img)
        targets.append(target)
    return images, targets

def get_split_image_and_annotation_paths(cfg):
    with open(f'{cfg["project_root"]}/data/annotations/trainval.txt') as f:
        trainval_names = [line.strip().split()[0] for line in f if line.strip()]

    with open(f'{cfg["project_root"]}/data/annotations/test.txt') as f:
        test_names = [line.strip().split()[0] for line in f if line.strip()]

    train_names, val_names = train_test_split(trainval_names, test_size=0.2, random_state=42, shuffle=True)

    train_imgs_paths, train_xmls_paths = get_paths(train_names, cfg)
    val_imgs_paths, val_xmls_paths     = get_paths(val_names, cfg)
    test_imgs_paths, test_xmls_paths   = get_paths(test_names, cfg)

    return (train_imgs_paths, train_xmls_paths), (val_imgs_paths, val_xmls_paths), (test_imgs_paths, test_xmls_paths)

def train_one_epoch(model, optimizer, loader, device):
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        # device 이동
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # forward + loss 계산
        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        # backward + step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate_one_epoch(
    model,
    loader: torch.utils.data.DataLoader,
    device: torch.device,
    score_thresh: float = 0.5
) -> tuple[float, float, dict]:
    """
    평가 루프: Loss, Mean IoU, mAP를 함께 계산

    Returns:
        mean_loss (float): 배치당 평균 loss
        mean_iou  (float): 이미지당 best(pred vs. gt) IoU 평균
        map_dict  (dict) : {'map', 'map_50', 'map_75', …} 형태의 mAP 결과
    """
    model.eval()
    total_loss = 0.0
    ious = []

    # COCO-style mAP (IoU 0.50:0.95)
    metric = MeanAveragePrecision(iou_type='bbox').to(device)

    with torch.no_grad():
        for images, targets in loader:
            # 1) device 옮기기
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # 2) Loss 계산
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            total_loss += loss.item()

            # 3) 예측 실행 (targets 없이)
            outputs = model(images)

            # 4) 각 이미지마다 IoU와 mAP 업데이트
            for tgt, out in zip(targets, outputs):

                boxes  = out['boxes'].to(device)
                scores = out['scores'].to(device)
                labels = out['labels'].to(device)

                # score threshold 적용
                keep = scores >= score_thresh
                if keep.sum() == 0:
                    # 예측 박스가 아예 없으면 건너뜀
                    continue

                # best prediction 하나 선택
                kept_boxes  = boxes[keep]
                kept_scores = scores[keep]
                best_idx    = kept_scores.argmax().item()
                pred_box    = kept_boxes[best_idx].unsqueeze(0)  # shape [1,4]

                # GT 박스
                gt_boxes = tgt['boxes'].to(device)  # shape [M,4]

                # IoU 계산
                iou_matrix = box_iou(pred_box, gt_boxes)
                ious.append(iou_matrix.max().item())

                # mAP 업데이트
                preds = {
                    'boxes':  pred_box.to(device),
                    'scores': kept_scores[best_idx].unsqueeze(0).to(device),
                    'labels': labels[keep][best_idx].unsqueeze(0).to(device)
                }
                gts = {
                    'boxes':  gt_boxes.to(device),
                    'labels': tgt['labels'].to(device)
                }
                metric.update([preds], [gts])

    # 최종 지표 정리
    mean_loss = total_loss / len(loader)
    mean_iou  = float(np.mean(ious)) if ious else 0.0
    map_dict  = metric.compute()  # e.g. {'map':..., 'map_50':..., 'map_75':..., ...}

    return mean_loss, mean_iou, map_dict

def visualize_grid(model, image_paths, class_map, device, cols=2):
    N = len(image_paths)
    rows = math.ceil(N / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
    axes_flat = axes.flatten() if isinstance(axes, np.ndarray) else [axes]
    idx_to_class = {v: k for k, v in class_map.items()}

    for ax, img_path in zip(axes_flat, image_paths):
        img = Image.open(img_path).convert('RGB')
        tensor = F.to_tensor(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(tensor)[0]
            boxes = out['boxes'].to(device)
            labels = out['labels'].to(device)
            scores = out['scores'].to(device)

            if scores.numel() > 0:
                best = scores.argmax().item()
                best_box = boxes[best:best + 1] # 코드 일관성 위해서 슬라이싱
                best_label = labels[best:best + 1]
                best_score = scores[best:best + 1]

                xmin, ymin, xmax, ymax = best_box.squeeze().tolist()
                rect = patches.Rectangle(
                    (xmin, ymin),
                    width=xmax - xmin,
                    height=ymax - ymin,
                    linewidth=2,
                    edgecolor='red',
                    facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(
                    xmin,
                    ymin - 5,
                    f"{idx_to_class[int(best_label)]}:{best_score.item():.2f}",
                    color='white',
                    fontsize=8,
                    backgroundcolor='red'
                )
        ax.imshow(img)
        ax.axis('off')

    # Remove unused axes
    for ax in axes_flat[N:]:
        ax.remove()
    plt.tight_layout()
    plt.show()

def compute_mean_iou(model, image_paths, xml_paths, class_map,
device: torch.device, score_thresh: float = 0.5) -> float:
    ious = []
    model.eval()
    with torch.no_grad():
        for img_path, xml_path in zip(image_paths, xml_paths):
            img = Image.open(img_path).convert('RGB')
            out = model(F.to_tensor(img).unsqueeze(0).to(device))[0]
            boxes, scores = out['boxes'].to(device), out['scores'].to(device)
            keep = scores >= score_thresh
            if keep.sum() == 0:
                continue
            best = scores[keep].argmax().item()
            pred_box = boxes[keep][best].unsqueeze(0)

            gt_boxes, gt_labels = parse_voc_xml(xml_path, class_map)
            if len(gt_boxes) == 0:
                continue
            gt = torch.tensor(gt_boxes, dtype=torch.float32)
            iou_matrix = box_iou(pred_box, gt)
            ious.append(iou_matrix.max().item())
    return float(np.mean(ious)) if ious else 0.0

def evaluate_map(model, image_paths, xml_paths, class_map, device):
    metric = MeanAveragePrecision(iou_type='bbox').to(device)
    model.eval()
    metric.reset()
    with torch.no_grad():
        for img_path, xml_path in zip(image_paths, xml_paths):
            img = Image.open(img_path).convert('RGB')
            out = model(F.to_tensor(img).unsqueeze(0).to(device))[0]
            # prediction
            preds = {
                'boxes': out['boxes'].to(device),
                'scores': out['scores'].to(device),
                'labels': out['labels'].to(device)
            }
            # ground-truth
            gt_boxes, gt_labels = parse_voc_xml(xml_path, class_map)
            gts = {
                'boxes': torch.tensor(gt_boxes, dtype=torch.float32),
                'labels': torch.tensor(gt_labels, dtype=torch.int64)
            }
            metric.update([preds], [gts])
    return metric.compute()

def main(cfg):
    # --- 설정 및 장치 ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs("checkpoints", exist_ok=True)
    set_seed(42)

    # --- 클래스 맵핑 & 모델 생성 ---
    class_map   = get_class_map(f'{cfg["project_root"]}/data/annotations/xmls')
    num_classes = len(class_map) + 1  # +1 background
    model = get_ssd_model(num_classes, pretrained=True, device=device)

    # --- Optimizer & Scheduler ---
    optimizer    = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4)
    lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

    # --- 데이터셋 분할 ---
    (train_images, train_xmls), (val_images, val_xmls), (test_images, test_xmls) = get_split_image_and_annotation_paths(cfg)

    # --- DataLoader ---
    train_ds = PetFaceDataset(train_images, train_xmls, class_map)
    val_ds   = PetFaceDataset(val_images,   val_xmls,   class_map)
    test_ds  = PetFaceDataset(test_images,  test_xmls,  class_map)

    g = torch.Generator()
    g.manual_seed(42)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, worker_init_fn=seed_worker, num_workers=4, collate_fn=collate_fn, generator=g, pin_memory=True, prefetch_factor=2)
    val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, collate_fn=collate_fn, worker_init_fn=seed_worker)
    test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2, collate_fn=collate_fn, worker_init_fn=seed_worker)

    # --- 학습 루프 ---
    best_val_loss = float("inf")
    num_epochs    = 20

    for epoch in range(1, num_epochs+1):
        train_mean_loss = train_one_epoch(model, optimizer, train_loader, device)
        val_mean_loss, val_mean_iou, val_mAP   = evaluate_one_epoch(model, val_loader,   device)
        test_mean_loss, test_mean_iou, test_mAP = evaluate_one_epoch(model, test_loader, device)
        print('---'*20)
        print(
            f"Epoch {epoch}/{num_epochs} - "
            f"Train Loss: {train_mean_loss:.4f}, "
            f"Val Loss: {val_mean_loss:.4f}, "
            f"Val mIoU: {val_mean_iou:.4f}, "
            f"Val mAP: {val_mAP['map']:.4f}, "
            f"Test Loss: {test_mean_loss:.4f}, "
            f"Test mIoU: {test_mean_iou:.4f}, "
            f"Test mAP: {test_mAP['map']:.4f}"
            )

        # 스케줄러 스텝
        lr_scheduler.step()

        # 베스트 모델 저장
        if val_mean_loss < best_val_loss:
            torch.save(model.state_dict(), "checkpoints/best_model.pth")
            best_val_loss = val_mean_loss
            print(f"Best model saved with validation loss: {best_val_loss:.4f}")
    # 시각화
    visualize_grid(model, val_images[:4], class_map, device, cols=2)
    visualize_grid(model, test_images[:4], class_map, device, cols=2)

In [4]:
def run_train(cfg):
    main(cfg)

cfg = {
    'project_root': '/content/drive/Othercomputers/내 MacBook Air/Documents/ai_project/mission7'
}
run_train(cfg)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:04<00:00, 127MB/s]
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 1.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/fetch.py", line 52, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "<ipython-input-3-62190b22b59e>", line 90, in __getitem__
    boxes, labels = parse_voc_xml(self.xml_paths[idx], self.class_map)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<ipython-input-3-62190b

In [ ]:
runtime.unassign()